# R2E-RoboLab v2 â€” GRPO Training Notebook
**Train Qwen2.5-1.5B-Instruct to reason before acting in robotic manipulation tasks**

## Setup Instructions
1. Open this notebook in Google Colab (Runtime â†’ T4 GPU)
2. Add your HuggingFace token: click ðŸ”‘ **Secrets** in left sidebar â†’ add `HF_TOKEN`
3. Run all cells in order

Expected training time: ~90 minutes on T4

In [ ]:
# Cell 1: Install dependencies
!pip install -q unsloth trl transformers accelerate peft datasets openai python-dotenv modelscope
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
print('Dependencies installed.')

In [ ]:
# Cell 2: Unzip the codebase
!unzip -q -o repo.zip -d /content/R2E-RoboLab-Robotic-Reasoning-Experimentation-Lab
import sys
sys.path.insert(0, '/content/R2E-RoboLab-Robotic-Reasoning-Experimentation-Lab')
print('Repo unzipped and added to path.')

In [ ]:
# Cell 3: Load HF token from Colab Secrets
from google.colab import userdata
import os
HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
print(f'Token loaded: {HF_TOKEN[:8]}...')

In [ ]:
# Cell 4: Load model with Unsloth 4-bit quantization
import os
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'
from unsloth import FastLanguageModel
import torch

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
MAX_SEQ_LEN = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    token=HF_TOKEN,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing=True,
    random_state=42,
)
print(f'Model loaded: {MODEL_NAME}')

In [ ]:
# Cell 5: Generate training dataset from the environment
import asyncio
from r2e_env.environment import R2EEnv
from agents import ReasoningAgent, RandomAgent, GreedyAgent
from datasets import Dataset

SYSTEM_PROMPT = """You are an AI agent debugging a robotic arm precision insertion task.
Hidden physical properties: friction_level, alignment_error, stiffness.
You MUST think step-by-step before every action.

Response format:
<think>
[reasoning about physical state and safe action]
</think>
Action: [one of: insert, adjust_left, adjust_right, increase_force, probe_friction, probe_alignment, probe_stiffness, commit_solution]

CRITICAL: NEVER use increase_force if friction=HIGH and stiffness=COMPLIANT (causes JAM).
Always probe before acting. Commit only when position>=1.0 and no failure."""

async def collect_episodes(n_easy=100, n_medium=80, n_hard=60):
    """Collect training prompts from oracle/reasoning agent episodes."""
    prompts = []
    agent = ReasoningAgent()

    task_seeds = [
        ('easy', n_easy),
        ('medium', n_medium),
        ('hard', n_hard),
    ]

    for task, n in task_seeds:
        for seed in range(n):
            env = R2EEnv()
            obs = await env.reset(task=task, seed=seed)
            probed = set()
            done = False

            while not done:
                obs_dict = obs.model_dump()
                known = obs_dict.get('known_variables', {})
                known_str = ', '.join(f'{k}={v}' for k, v in known.items()) or 'none yet'

                user_msg = f"""Step {obs_dict['step_count']+1} | Task: {task} | Phase: {obs_dict['phase']}

Observation:
  position:            {obs_dict['position']:.3f}
  force_feedback:      {obs_dict['force_feedback']:.3f}
  lateral_instability: {obs_dict['lateral_instability']:.3f}
  failure_signal:      {obs_dict['failure_signal']}
  last_action:         {obs_dict['last_action']}

Known properties: {known_str}

Think and choose your action:"""

                action = agent.act(obs, probed)
                prompts.append({
                    'task': task,
                    'seed': seed,
                    'system': SYSTEM_PROMPT,
                    'user': user_msg,
                    'action': action.action,
                    'reasoning': action.reasoning,
                })

                obs, reward, done, info = await env.step(action)

        print(f'Collected {task}: {sum(1 for p in prompts if p["task"]==task)} prompts')

    return prompts

prompts = await collect_episodes()
print(f'Total training prompts: {len(prompts)}')

In [ ]:
# Cell 6: Format dataset for GRPO training
from datasets import Dataset

def format_prompt(sample):
    """Format as chat template for GRPO."""
    messages = [
        {'role': 'system', 'content': sample['system']},
        {'role': 'user', 'content': sample['user']},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return {'prompt': text, 'task': sample['task'], 'seed': sample['seed'],
            'target_action': sample['action']}

dataset = Dataset.from_list(prompts)
dataset = dataset.map(format_prompt)
dataset = dataset.shuffle(seed=42)

# Split train/eval
split = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = split['train']
eval_ds = split['test']

print(f'Train: {len(train_ds)} | Eval: {len(eval_ds)}')

In [ ]:
# Cell 7: Define GRPO reward functions
import re
import asyncio
from r2e_env.environment import R2EEnv
from r2e_env.models import R2EAction

VALID_ACTIONS = [
    'insert', 'adjust_left', 'adjust_right', 'increase_force',
    'probe_friction', 'probe_alignment', 'probe_stiffness', 'commit_solution'
]

def parse_response(text):
    """Extract reasoning and action from LLM response."""
    think_match = re.search(r'<think>(.*?)</think>', text, re.DOTALL | re.IGNORECASE)
    reasoning = think_match.group(1).strip() if think_match else ''
    action_match = re.search(r'Action:\s*([a-z_]+)', text, re.IGNORECASE)
    if action_match:
        action = action_match.group(1).lower()
        if action in VALID_ACTIONS:
            return reasoning, action
    for a in VALID_ACTIONS:
        if a in text.lower():
            return reasoning, a
    return reasoning, 'probe_friction'


# Reward 1: Format correctness
def format_reward(completions, **kwargs):
    rewards = []
    for c in completions:
        text = c[0]['content'] if isinstance(c, list) else c
        has_think = bool(re.search(r'<think>.*?</think>', text, re.DOTALL))
        has_action = bool(re.search(r'Action:\s*([a-z_]+)', text, re.IGNORECASE))
        r = 0.0
        if has_think: r += 0.3
        if has_action: r += 0.3
        reasoning, action = parse_response(text)
        if action in VALID_ACTIONS: r += 0.4
        rewards.append(r)
    return rewards


# Reward 2: Reasoning quality (no deceptive actions without checking)
def reasoning_quality_reward(completions, **kwargs):
    rewards = []
    for c in completions:
        text = c[0]['content'] if isinstance(c, list) else c
        reasoning, action = parse_response(text)
        r = 0.0
        t = reasoning.lower()

        # Reward probing intent in early steps
        if 'probe' in t: r += 0.15
        # Reward caution about increase_force
        if 'increase_force' in t and any(w in t for w in ['danger', 'risky', 'jam', 'avoid', 'careful']):
            r += 0.20
        # Reward mentioning friction or stiffness
        if 'friction' in t: r += 0.10
        if 'stiffness' in t or 'compliant' in t: r += 0.10
        # Penalize committing without mentioning position
        if action == 'commit_solution' and 'position' not in t: r -= 0.30
        rewards.append(max(-1.0, min(1.0, r)))
    return rewards


# Reward 3: Safety â€” never use increase_force when known to be dangerous
def safety_reward(completions, **kwargs):
    rewards = []
    for c in completions:
        text = c[0]['content'] if isinstance(c, list) else c
        reasoning, action = parse_response(text)
        t = text.lower()

        # If the prompt contains known friction=high and stiffness=compliant,
        # penalize using increase_force
        is_dangerous_context = ('friction=high' in t or 'friction: high' in t) and \
                               ('stiffness=compliant' in t or 'compliant' in t)
        if is_dangerous_context and action == 'increase_force':
            rewards.append(-1.0)  # Strong penalty for walking into the trap
        elif action in VALID_ACTIONS:
            rewards.append(0.5)
        else:
            rewards.append(-0.5)
    return rewards

print('Reward functions defined.')

In [ ]:
# Cell 8: Configure and run GRPO training
from trl import GRPOTrainer, GRPOConfig

training_args = GRPOConfig(
    output_dir='r2e_grpo_output',
    num_generations=4,           # 4 candidates per prompt
    max_new_tokens=300,          # enough for <think> + Action:
    temperature=0.8,
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    warmup_ratio=0.1,
    logging_steps=5,
    save_steps=50,
    eval_steps=25,
    evaluation_strategy='steps',
    report_to='none',            # Change to 'wandb' if you have W&B
    fp16=True,
    seed=42,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        format_reward,
        reasoning_quality_reward,
        safety_reward,
    ],
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
)

print('Starting GRPO training...')
print(f'Train samples: {len(train_ds)} | Steps: ~{len(train_ds)//training_args.per_device_train_batch_size * 3}')
trainer.train()

In [ ]:
# Cell 9: Plot training curves
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import json

# Extract training log
log = trainer.state.log_history
train_steps = [x['step'] for x in log if 'loss' in x]
train_rewards = [x.get('reward', x.get('train_reward', 0)) for x in log if 'loss' in x]
train_loss = [x['loss'] for x in log if 'loss' in x]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(train_steps, train_rewards, color='#2ecc71', linewidth=2)
ax1.set_xlabel('Training Step')
ax1.set_ylabel('Mean Reward')
ax1.set_title('GRPO Reward Curve\nR2E-RoboLab v2 â€” Qwen2.5-1.5B')
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='0.5 baseline')
ax1.legend()

ax2.plot(train_steps, train_loss, color='#e74c3c', linewidth=2)
ax2.set_xlabel('Training Step')
ax2.set_ylabel('Loss')
ax2.set_title('Training Loss Curve\nR2E-RoboLab v2 â€” Qwen2.5-1.5B')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved as training_curves.png')

# Save log to JSON
with open('training_log.json', 'w') as f:
    json.dump(log, f, indent=2)
print('Training log saved.')

In [ ]:
# Cell 10: Before vs After comparison
# Load the untrained base model for comparison
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

TEST_OBSERVATION = """Step 3 | Task: hard | Phase: investigation

Observation:
  position:            0.000
  force_feedback:      0.900
  lateral_instability: 0.050
  failure_signal:      none
  last_action:         probe_friction

Known properties: friction=high

Think and choose your action:"""

messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': TEST_OBSERVATION},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

# Generate with TRAINED model
FastLanguageModel.for_inference(model)
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=300, temperature=0.1, do_sample=True)
trained_response = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print('='*60)
print('TEST OBSERVATION (friction=HIGH, known after probing):')
print('Expected safe action: probe_stiffness or insert (NOT increase_force)')
print('='*60)
print('TRAINED MODEL RESPONSE:')
print(trained_response)
print('='*60)

In [ ]:
# Cell 11: Save and push trained model to HuggingFace Hub
from huggingface_hub import login
login(token=HF_TOKEN)

REPO_ID = 'monika-10333/r2e-robolab-qwen2.5-1.5b-grpo'

model.save_pretrained('r2e_trained_model')
tokenizer.save_pretrained('r2e_trained_model')

# Push to hub
model.push_to_hub(REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(REPO_ID, token=HF_TOKEN)

print(f'Model pushed to: https://huggingface.co/{REPO_ID}')

In [ ]:
# Cell 12: Download artifacts (training_curves.png, training_log.json)
from google.colab import files
files.download('training_curves.png')
files.download('training_log.json')
print('Download initiated. Save these files to results/ in your project repo.')